# FastSurfer inputs: training on Kaggle (2 inputs × 3 held-out hospitals)

Section 7 of `notebooks/fastsurfer_colab.ipynb`, on Kaggle: same setting as the conform baseline (seed 42, no augmentation, no scheduler, 50 epochs max, batch size 8, `best-mae` checkpoint). The two T4 work in parallel: `fastsurfer_nu` on GPU 0, `fastsurfer_brain` on GPU 1, 3 runs each (~1 h per run, so ~3-4 h in all).

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **on**.

**Input:** the output of the FastSurfer notebook (Add Input → Your Work → the FastSurfer notebook, last version: 563 / 563 subjects).

**Run in the background:** Save Version → *Save & Run All (Commit)*. Results are in the Output tab (`outputs/site_shift_test-<site>_fastsurfer_<nu|brain>/`). New runs stop starting after 10 h; if a session ends before the 6 runs are done, run a new version with **this notebook's output added as input**: finished runs are copied and skipped.

Afterwards, put the 6 `outputs/site_shift_test-*_fastsurfer_*` folders in `MyDrive/test-clinicadl/outputs` to compare with conform and t1-linear (section 8 of the Colab notebook).

## 1. Scripts, clinicadl, images and runs already done

In [ ]:
import os, glob, shutil, time
import pandas as pd
from IPython.display import Image, display
os.environ['MPLBACKEND'] = 'Agg'
T0 = time.time()

WORK = '/tmp/test-clinicadl'
OUT = '/kaggle/working/outputs'                       # saved as the notebook output
REPO = 'https://github.com/Ayoub-TAREK/cross-site-brain-age.git'
if not os.path.exists(WORK):
    !git clone -q {REPO} {WORK}
!git -C {WORK} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q clinicadl==2.0.0 2>&1 | grep -v -i warning | tail -n 3
!python -c "import torch, clinicadl; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())"

# FastSurfer images: the input folder with the most finished subjects, copied to the local disk
pattern = '/subjects/*/*/fastsurfer/*_desc-brain_res-2x2x2_T1w.nii.gz'
caps = sorted({os.path.dirname(f) for f in glob.glob('/kaggle/input/**/caps_fastsurfer_2mm/participants_sessions_age.tsv', recursive=True)},
              key=lambda d: len(glob.glob(d + pattern)))
assert caps, 'caps_fastsurfer_2mm not found: add the output of the FastSurfer notebook as input'
n = len(glob.glob(caps[-1] + pattern))
print(caps[-1], ':', n, 'subjects')
assert n == 563, f'{n} / 563 subjects: use the last version of the FastSurfer notebook as input'
LOCAL = f'{WORK}/data/IXI/caps_fastsurfer_2mm'
if not os.path.exists(LOCAL):
    shutil.copytree(caps[-1], LOCAL)

# outputs/ of the scripts -> /kaggle/working/outputs; finished runs found in the inputs are copied (then skipped)
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(OUT, f'{WORK}/outputs')
for f in glob.glob('/kaggle/input/**/outputs/site_shift_test-*/predictions_best-mae.tsv', recursive=True):
    run = os.path.dirname(f)
    if not os.path.exists(f'{OUT}/{os.path.basename(run)}/predictions_best-mae.tsv'):
        shutil.copytree(run, f'{OUT}/{os.path.basename(run)}', dirs_exist_ok=True)
        print('finished run copied from the inputs:', os.path.basename(run))

## 2. Check the inputs (one image per hospital)

In [ ]:
%cd {WORK}
!python scripts/train_ixi_site_shift.py --data fastsurfer_nu --check
!python scripts/train_ixi_site_shift.py --data fastsurfer_brain --check
for p in sorted(glob.glob(f'{OUT}/site_shift_test-IOP_fastsurfer_*/check_*.png')):
    print(os.path.basename(os.path.dirname(p)))
    display(Image(p))

## 3. Training: `nu` on GPU 0 and `brain` on GPU 1, in parallel

One line per epoch and the prediction summary of each run, prefixed by the input. Finished runs are skipped; an unfinished run is restarted from the beginning. No new run starts after 10 h.

In [ ]:
import subprocess
DEADLINE = T0 + 10 * 3600
WORKER = '''
cd {work}
for site in HH Guys IOP; do
  run=outputs/site_shift_test-${{site}}_{data}
  if [ -f $run/predictions_best-mae.tsv ]; then echo "already done, skipped: $run"; continue; fi
  if [ $(date +%s) -gt {deadline} ]; then echo "time budget reached: $run not started"; continue; fi
  rm -rf $run/maps $run/split $run/train_sites.tsv $run/out_of_site.tsv
  echo "===== $run ($(date +%H:%M)) ====="
  flags="--data {data} --test-site $site --seed 42"
  python scripts/train_ixi_site_shift.py $flags --epochs 50 --batch-size 8 2>&1 | grep --line-buffered -E "seed :|Epoch [0-9]+ completed|stopped|Error|Traceback"
  python scripts/predict_ages.py $flags --checkpoint best-mae 2>&1 | grep -v -i warning | tail -n 12
done
echo "===== {data} finished ($(date +%H:%M)) ====="
'''
procs, logs = {}, {}
for gpu, data in enumerate(['fastsurfer_nu', 'fastsurfer_brain']):
    logs[data] = f'/kaggle/working/train_{data}.log'
    script = WORKER.format(work=WORK, data=data, deadline=int(DEADLINE))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1')
    procs[data] = subprocess.Popen(['bash', '-c', script], env=env, stdout=open(logs[data], 'w'), stderr=subprocess.STDOUT)

printed = {d: 0 for d in procs}
def show_new_lines():
    for d, path in logs.items():
        lines = open(path).read().splitlines()
        for line in lines[printed[d]:]:
            print(f'[{d[11:]:5}] {line}', flush=True)
        printed[d] = len(lines)
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    show_new_lines()
show_new_lines()
print(f'\ntotal: {(time.time() - T0) / 3600:.1f} h')

## 4. Results

Out-of-site MAE (in-site in parentheses), out-of-site bias and correlation, `best-mae`, seed 42. Conform and t1-linear rows appear only if their runs are in the inputs; otherwise compare in section 8 of the Colab notebook.

In [ ]:
rows = []
for data in ['conform', 'fastsurfer_nu', 'fastsurfer_brain', 't1linear']:
    for site in ['IOP', 'HH', 'Guys']:
        f = f'{OUT}/site_shift_test-{site}_{data}/predictions_best-mae.tsv'
        if not os.path.exists(f):
            continue
        df = pd.read_csv(f, sep='\t')
        ins, oos = df[df.group == 'insite'], df[df.group == 'outofsite']
        rows.append(dict(data=data, site=site,
                         mae=f'{oos.error.abs().mean():.1f} ({ins.error.abs().mean():.1f})',
                         bias=round(oos.error.mean(), 1), r=round(oos.AgeMRI.corr(oos.predicted), 2)))
res = pd.DataFrame(rows)
print(len(glob.glob(f'{OUT}/site_shift_test-*_fastsurfer_*/predictions_best-mae.tsv')), '/ 6 FastSurfer runs done')
for col in ['mae', 'bias', 'r']:
    print(f'\n{col}')
    print(res.pivot(index='data', columns='site', values=col).to_string())
!du -sh {OUT}